# flowgram gallery

Each cell builds a diagram and shows it. The same code is in the matching script in this folder.

## A linear pipeline: five steps in a row, each one feeding the next.

``uniform`` gives every step the size of the largest, so the row reads as one
even band. Run this file to write ``linear_pipeline.png`` next to it.

In [ ]:
from itertools import pairwise

from flowgram import Box, Diagram, Edge, Row

spec = Diagram(
    title="Nightly training run",
    subtitle="Each step starts when the one before it succeeds",
    root=Row(
        gap=1,
        uniform=True,
        children=[
            Box(id="ingest", label="Ingest", subtitle="daily batch", role="muted"),
            Box(id="validate", label="Validate", subtitle="schema, ranges"),
            Box(id="transform", label="Transform", subtitle="join, aggregate"),
            Box(id="train", label="Train", subtitle="gradient boosting"),
            Box(id="publish", label="Publish", subtitle="registry", role="accent"),
        ],
    ),
    edges=[
        Edge(src=src, dst=dst)
        for src, dst in pairwise(
            ["ingest", "validate", "transform", "train", "publish"]
        )
    ],
)

spec

## A branching model flow: two models fork from shared features and meet again.

Boxes stacked in a Column fan edges out and back in, and labelled edges
mark the two outcomes. ``exit="bottom"`` sends the "worse" edge out of its own
side, so the two outcome labels do not share a line. Run this file to write
``branching_flow.png`` next to it.

In [ ]:
from flowgram import Box, Column, Diagram, Edge, Row

spec = Diagram(
    title="Champion and challenger",
    subtitle="A new model ships only if it beats the current one",
    root=Row(
        gap=2,
        children=[
            Box(id="features", label="Features", subtitle="shared", role="muted"),
            Column(
                gap=1,
                children=[
                    Box(id="champion", label="Champion", subtitle="in production"),
                    Box(id="challenger", label="Challenger", role="accent"),
                ],
            ),
            Box(id="compare", label="Compare", subtitle="hold-out set"),
            Column(
                gap=1,
                children=[
                    Box(id="ship", label="Ship", role="positive"),
                    Box(id="keep", label="Keep champion", role="muted"),
                ],
            ),
        ],
    ),
    edges=[
        Edge(src="features", dst="champion"),
        Edge(src="features", dst="challenger"),
        Edge(src="champion", dst="compare", style="dashed"),
        Edge(src="challenger", dst="compare"),
        Edge(src="compare", dst="ship", label="better"),
        Edge(src="compare", dst="keep", label="worse", style="dashed", exit="bottom"),
    ],
)

spec

## A nested layout: columns inside a row inside a column, in the mono theme.

Containers nest to any depth. ``align="start"`` tops the service columns
against each other, and edges reach across the nesting. Run this file to
write ``nested_layout.png`` next to it.

In [ ]:
from flowgram import Box, Column, Diagram, Edge, Row, Theme


def service(name: str, *parts: str) -> Column:
    """Return a service box over the boxes it owns."""
    owned = [Box(id=part, label=part.title(), role="muted") for part in parts]
    return Column(gap=0.5, children=[Box(id=name, label=name.title()), *owned])


spec = Diagram(
    title="Order platform",
    theme=Theme.mono(),
    root=Column(
        gap=1.5,
        children=[
            Row(
                gap=1, children=[Box(id="web", label="Web"), Box(id="app", label="App")]
            ),
            Box(id="gateway", label="API gateway", role="accent"),
            Row(
                gap=1.5,
                align="start",
                children=[
                    service("accounts", "users"),
                    service("orders", "queue", "billing"),
                    service("search", "index"),
                ],
            ),
        ],
    ),
    edges=[
        *(Edge(src=client, dst="gateway") for client in ("web", "app")),
        *(Edge(src="gateway", dst=name) for name in ("accounts", "orders", "search")),
        *(Edge(src=a, dst=b) for a, b in [("accounts", "users"), ("orders", "queue")]),
        Edge(src="queue", dst="billing"),
        Edge(src="search", dst="index"),
    ],
)

spec